# 📖 Notebook 3: Back-of-Envelope Estimation Practice

This is where everything comes together. In system design interviews, you'll be asked to
**estimate capacity** for a system before you design it. This shows the interviewer that you
think about scale methodically, not by guessing.

## Learning Objectives

By the end of this notebook, you'll be able to:
- Apply a repeatable framework for any estimation question
- Estimate QPS, storage, and bandwidth for Twitter, YouTube, and Uber
- Show your work clearly (the process matters more than the exact number)
- Avoid common estimation mistakes

## 🛠️ Setup

No Docker needed for this notebook — it's all math and Python!

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
# Helper functions for estimation

def fmt(n):
    """Format a large number in human-readable form."""
    if n >= 1e15:
        return f"{n/1e15:.1f}P"
    elif n >= 1e12:
        return f"{n/1e12:.1f}T"
    elif n >= 1e9:
        return f"{n/1e9:.1f}B"
    elif n >= 1e6:
        return f"{n/1e6:.1f}M"
    elif n >= 1e3:
        return f"{n/1e3:.1f}K"
    else:
        return f"{n:.0f}"

def fmt_bytes(b):
    """Format bytes in human-readable form."""
    if b >= 1e15:
        return f"{b/1e15:.1f} PB"
    elif b >= 1e12:
        return f"{b/1e12:.1f} TB"
    elif b >= 1e9:
        return f"{b/1e9:.1f} GB"
    elif b >= 1e6:
        return f"{b/1e6:.1f} MB"
    elif b >= 1e3:
        return f"{b/1e3:.1f} KB"
    else:
        return f"{b:.0f} B"

def to_qps(daily):
    return daily / 86_400

def print_divider():
    print("─" * 60)

print("✅ Helper functions loaded!")
print()
print("📋 Reference: Key numbers to keep in mind")
print("─" * 50)
print("  Seconds in a day:          86,400 ≈ 100K")
print("  Single Postgres QPS:       ~10,000 (reads)")
print("  Single Redis QPS:          ~100,000")
print("  Modern server RAM:         256-512 GB")
print("  Modern server SSD:         up to 60 TB")
print("  Network per instance:      25 Gbps")

## 🧩 The Estimation Framework

Every estimation follows the same 5 steps. Practice this framework until it's automatic:

```
┌────────────────────────────────────────────────────────┐
│  Step 1: CLARIFY                                       │
│  What exactly are we estimating? State assumptions.    │
├────────────────────────────────────────────────────────┤
│  Step 2: USERS & ACTIVITY                              │
│  MAU → DAU → actions per user per day                  │
├────────────────────────────────────────────────────────┤
│  Step 3: QPS                                           │
│  daily_actions / 86,400 = avg QPS                      │
│  avg QPS × 3 = peak QPS                                │
├────────────────────────────────────────────────────────┤
│  Step 4: STORAGE                                       │
│  items/day × bytes/item × 365 × retention_years        │
├────────────────────────────────────────────────────────┤
│  Step 5: BANDWIDTH                                     │
│  QPS × response_size_bytes × 8 = bits/sec              │
└────────────────────────────────────────────────────────┘
```

💡 **In interviews, always show your work.** Say each step out loud.
The interviewer cares more about your process than the exact answer.

---

## 🐦 Practice 1: Twitter

Let's estimate the capacity requirements for a Twitter-like system.

### Step 1: Clarify
- We're designing the **tweet read/write** system (not DMs, ads, search, etc.)
- Focus on: tweet creation, timeline reads, fan-out
- Assume: text tweets only (no media for simplicity)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 🐦 TWITTER ESTIMATION
# ═══════════════════════════════════════════════════════════

print("🐦 Twitter Capacity Estimation")
print("=" * 60)
print()

# ── Step 2: Users & Activity ──
print("Step 2: Users & Activity")
print_divider()

mau = 500_000_000              # 500M monthly active users
dau_ratio = 0.4                # 40% are daily active
dau = int(mau * dau_ratio)

tweets_per_user_per_day = 0.5  # Most users lurk; avg 1 tweet per 2 days
timeline_views_per_day = 20    # Each user opens/scrolls their feed ~20 times
tweets_per_timeline = 50       # Each timeline load shows ~50 tweets

print(f"  MAU:                 {fmt(mau)}")
print(f"  DAU (40% of MAU):    {fmt(dau)}")
print(f"  Tweets/user/day:     {tweets_per_user_per_day}")
print(f"  Timeline views/day:  {timeline_views_per_day}")
print(f"  Tweets per view:     {tweets_per_timeline}")
print()

# ── Step 3: QPS ──
print("Step 3: QPS")
print_divider()

# Writes: tweet creation
tweets_per_day = dau * tweets_per_user_per_day
write_qps_avg = to_qps(tweets_per_day)
write_qps_peak = write_qps_avg * 3

# Reads: timeline views (each view = 1 request that returns 50 tweets)
timeline_reads_per_day = dau * timeline_views_per_day
read_qps_avg = to_qps(timeline_reads_per_day)
read_qps_peak = read_qps_avg * 3

print(f"  Tweets created/day:    {fmt(tweets_per_day)}")
print(f"  Write QPS (avg):       {write_qps_avg:,.0f}")
print(f"  Write QPS (peak):      {write_qps_peak:,.0f}")
print()
print(f"  Timeline reads/day:    {fmt(timeline_reads_per_day)}")
print(f"  Read QPS (avg):        {read_qps_avg:,.0f}")
print(f"  Read QPS (peak):       {read_qps_peak:,.0f}")
print()
print(f"  Read/Write ratio:      {read_qps_avg/write_qps_avg:.0f}:1 (read-heavy!)")
print()

# ── Step 4: Storage ──
print("Step 4: Storage")
print_divider()

tweet_size_bytes = 500         # 280 chars + metadata (user_id, timestamp, etc.)
retention_years = 10

daily_storage = tweets_per_day * tweet_size_bytes
yearly_storage = daily_storage * 365
total_storage = yearly_storage * retention_years

print(f"  Tweet size:            {tweet_size_bytes} bytes")
print(f"  Tweets/day:            {fmt(tweets_per_day)}")
print(f"  Storage/day:           {fmt_bytes(daily_storage)}")
print(f"  Storage/year:          {fmt_bytes(yearly_storage)}")
print(f"  Total ({retention_years} years):      {fmt_bytes(total_storage)}")
print()
if total_storage < 10e12:
    print(f"  ✅ {fmt_bytes(total_storage)} fits on a single PostgreSQL!")
else:
    print(f"  ⚠️  {fmt_bytes(total_storage)} — may need partitioning or sharding")
print()

# ── Step 5: Bandwidth ──
print("Step 5: Bandwidth")
print_divider()

# Read bandwidth: each timeline response ≈ 50 tweets × 500B = 25KB
timeline_response_size = tweets_per_timeline * tweet_size_bytes
read_bandwidth_mbps = (read_qps_peak * timeline_response_size * 8) / 1e6

# Write bandwidth: each tweet ≈ 500B
write_bandwidth_mbps = (write_qps_peak * tweet_size_bytes * 8) / 1e6

print(f"  Timeline response:     {fmt_bytes(timeline_response_size)}")
print(f"  Read bandwidth (peak): {read_bandwidth_mbps:,.0f} Mbps ({read_bandwidth_mbps/1000:.1f} Gbps)")
print(f"  Write bandwidth (peak):{write_bandwidth_mbps:,.0f} Mbps")
print()

# ── Fan-out calculation (bonus) ──
print("Bonus: Fan-out")
print_divider()
avg_followers = 200
fanout_per_day = tweets_per_day * avg_followers
fanout_qps = to_qps(fanout_per_day)

print(f"  Avg followers/user:    {avg_followers}")
print(f"  Fan-outs/day:          {fmt(fanout_per_day)}")
print(f"  Fan-out QPS:           {fanout_qps:,.0f}")
print(f"  (Each tweet must be delivered to {avg_followers} follower timelines)")
print()

# ── Summary ──
print("📋 Twitter Summary")
print("=" * 60)
print(f"  Write QPS (peak):    {write_qps_peak:>10,.0f}  ← 1 Postgres can handle this")
print(f"  Read QPS (peak):     {read_qps_peak:>10,.0f}  ← Need Redis caching")
print(f"  Fan-out QPS:         {fanout_qps:>10,.0f}  ← Async processing needed")
print(f"  Total storage (10y): {fmt_bytes(total_storage):>10}  ← Fits ~1 server")
print(f"  Peak bandwidth:      {read_bandwidth_mbps/1000:>8.1f} Gbps ← 1 server (25 Gbps)")

---

## 📺 Practice 2: YouTube

YouTube is interesting because it's **bandwidth-dominated** (video streaming)
rather than QPS-dominated. The storage numbers are also massive.

### Step 1: Clarify
- We're estimating the **video upload and viewing** system
- Focus on: upload storage, viewing bandwidth, metadata QPS
- Assume: multiple quality levels (360p, 720p, 1080p)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 📺 YOUTUBE ESTIMATION
# ═══════════════════════════════════════════════════════════

print("📺 YouTube Capacity Estimation")
print("=" * 60)
print()

# ── Step 2: Users & Activity ──
print("Step 2: Users & Activity")
print_divider()

yt_mau = 2_000_000_000         # 2B monthly active
yt_dau_ratio = 0.4             # 40% daily active
yt_dau = int(yt_mau * yt_dau_ratio)

videos_watched_per_day = 5     # avg user watches 5 videos/day
avg_video_minutes = 7          # avg video length ~7 min

# Uploads: much smaller
uploaders_ratio = 0.001        # 0.1% of DAU upload content
uploads_per_uploader = 1       # 1 video per day for active uploaders

print(f"  MAU:                   {fmt(yt_mau)}")
print(f"  DAU:                   {fmt(yt_dau)}")
print(f"  Videos watched/user:   {videos_watched_per_day}")
print(f"  Avg video length:      {avg_video_minutes} min")
print(f"  Uploaders (0.1% DAU):  {fmt(int(yt_dau * uploaders_ratio))}")
print()

# ── Step 3: QPS ──
print("Step 3: QPS")
print_divider()

# Video views
total_views_per_day = yt_dau * videos_watched_per_day
view_qps_avg = to_qps(total_views_per_day)
view_qps_peak = view_qps_avg * 3

# Video uploads
total_uploads_per_day = int(yt_dau * uploaders_ratio * uploads_per_uploader)
upload_qps_avg = to_qps(total_uploads_per_day)

# Metadata reads (search, recommendations, etc.) — much higher than video views
metadata_reads_per_view = 5    # search, recommendation API, comments, etc.
metadata_qps = view_qps_peak * metadata_reads_per_view

print(f"  Video views/day:       {fmt(total_views_per_day)}")
print(f"  View QPS (avg):        {view_qps_avg:,.0f}")
print(f"  View QPS (peak):       {view_qps_peak:,.0f}")
print(f"  Upload QPS:            {upload_qps_avg:,.0f}")
print(f"  Metadata QPS (peak):   {metadata_qps:,.0f}")
print()

# ── Step 4: Storage ──
print("Step 4: Storage")
print_divider()

# Video storage (multiple quality levels)
avg_video_size_mb = 50         # compressed, single quality (720p)
quality_levels = 3             # 360p, 720p, 1080p
total_per_video_mb = avg_video_size_mb * quality_levels

daily_upload_storage = total_uploads_per_day * total_per_video_mb * 1e6  # bytes
yearly_upload_storage = daily_upload_storage * 365

# Metadata storage (much smaller)
metadata_per_video = 10_000    # 10 KB (title, description, tags, etc.)
daily_metadata = total_uploads_per_day * metadata_per_video
yearly_metadata = daily_metadata * 365

print(f"  Video uploads/day:     {fmt(total_uploads_per_day)}")
print(f"  Size per video:        {avg_video_size_mb} MB (per quality level)")
print(f"  Quality levels:        {quality_levels} (360p, 720p, 1080p)")
print(f"  Total per video:       {total_per_video_mb} MB")
print(f"  Video storage/day:     {fmt_bytes(daily_upload_storage)}")
print(f"  Video storage/year:    {fmt_bytes(yearly_upload_storage)}")
print()
print(f"  Metadata per video:    {fmt_bytes(metadata_per_video)}")
print(f"  Metadata/year:         {fmt_bytes(yearly_metadata)}")
print()

# ── Step 5: Bandwidth ──
print("Step 5: Bandwidth")
print_divider()

# Viewing bandwidth (the BIG number)
avg_bitrate_mbps = 5           # average streaming bitrate (720p)
concurrent_viewers = view_qps_peak  # simplified: each viewer = 1 stream

# More accurate: avg viewing time = 7 min, so at any moment:
avg_viewing_time_sec = avg_video_minutes * 60
views_per_sec = view_qps_peak
# Concurrent streams ≈ views_per_sec × avg_viewing_time_sec
concurrent_streams = views_per_sec * avg_viewing_time_sec

viewing_bandwidth_tbps = (concurrent_streams * avg_bitrate_mbps) / 1e6

# Upload bandwidth
upload_bandwidth_gbps = (upload_qps_avg * total_per_video_mb * 8) / 1e3

print(f"  Avg streaming bitrate:   {avg_bitrate_mbps} Mbps (720p)")
print(f"  Concurrent streams:      {fmt(concurrent_streams)}")
print(f"  Viewing bandwidth:       {viewing_bandwidth_tbps:,.0f} Tbps")
print(f"  Upload bandwidth:        {upload_bandwidth_gbps:,.0f} Gbps")
print()
print(f"  💡 {viewing_bandwidth_tbps:,.0f} Tbps is MASSIVE — this is why YouTube")
print(f"     uses thousands of CDN edge servers worldwide!")

# ── Summary ──
print()
print("📋 YouTube Summary")
print("=" * 60)
print(f"  View QPS (peak):       {view_qps_peak:>12,.0f}")
print(f"  Metadata QPS (peak):   {metadata_qps:>12,.0f}")
print(f"  Upload QPS:            {upload_qps_avg:>12,.0f}")
print(f"  Video storage/year:    {fmt_bytes(yearly_upload_storage):>12}")
print(f"  Viewing bandwidth:     {viewing_bandwidth_tbps:>10,.0f} Tbps")
print()
print("  Key insight: YouTube's challenge is BANDWIDTH, not QPS.")
print("  That's why CDNs are the #1 architectural decision.")

---

## 🚗 Practice 3: Uber

Uber is interesting because it's **write-heavy** (constant location updates)
and has **real-time requirements** (matching must happen in seconds).

### Step 1: Clarify
- We're estimating the **ride request and matching** system
- Focus on: location updates, ride matching, trip storage
- Assume: rides in ~500 cities worldwide

In [ ]:
# ═══════════════════════════════════════════════════════════
# 🚗 UBER ESTIMATION
# ═══════════════════════════════════════════════════════════

print("🚗 Uber Capacity Estimation")
print("=" * 60)
print()

# ── Step 2: Users & Activity ──
print("Step 2: Users & Activity")
print_divider()

uber_riders_mau = 100_000_000    # 100M monthly active riders
uber_rides_per_day = 25_000_000  # 25M rides/day globally
active_drivers = 5_000_000       # 5M active drivers

avg_ride_duration_min = 20       # average ride is 20 minutes
driver_online_hours = 8          # average driver is online 8 hours/day

print(f"  Monthly active riders: {fmt(uber_riders_mau)}")
print(f"  Rides per day:         {fmt(uber_rides_per_day)}")
print(f"  Active drivers:        {fmt(active_drivers)}")
print(f"  Avg ride duration:     {avg_ride_duration_min} min")
print(f"  Driver online hours:   {driver_online_hours} hrs/day")
print()

# ── Step 3: QPS ──
print("Step 3: QPS — Location Updates (the BIG number)")
print_divider()

# Drivers send location every 5 seconds while online
location_interval_sec = 5
updates_per_driver_per_day = (driver_online_hours * 3600) / location_interval_sec

total_location_updates_per_day = active_drivers * updates_per_driver_per_day
location_qps_avg = to_qps(total_location_updates_per_day)
location_qps_peak = location_qps_avg * 2  # less peaky than social media

print(f"  Location update interval:   Every {location_interval_sec} sec")
print(f"  Updates/driver/day:         {updates_per_driver_per_day:,.0f}")
print(f"  Total updates/day:          {fmt(total_location_updates_per_day)}")
print(f"  Location QPS (avg):         {location_qps_avg:,.0f}")
print(f"  Location QPS (peak):        {location_qps_peak:,.0f}")
print()

# Ride requests
print("Step 3: QPS — Ride Requests")
print_divider()

ride_request_qps = to_qps(uber_rides_per_day)
ride_request_peak = ride_request_qps * 3

# Each ride request triggers: matching, ETA calculation, pricing
api_calls_per_request = 5  # match, price, ETA, confirm, notify driver
total_api_qps = ride_request_peak * api_calls_per_request

print(f"  Rides/day:                  {fmt(uber_rides_per_day)}")
print(f"  Ride request QPS (avg):     {ride_request_qps:,.0f}")
print(f"  Ride request QPS (peak):    {ride_request_peak:,.0f}")
print(f"  API calls per request:      {api_calls_per_request}")
print(f"  Total API QPS (peak):       {total_api_qps:,.0f}")
print()

# ── Step 4: Storage ──
print("Step 4: Storage")
print_divider()

# Location data
location_update_bytes = 100     # lat, lng, timestamp, driver_id, speed
daily_location_bytes = total_location_updates_per_day * location_update_bytes
yearly_location_bytes = daily_location_bytes * 365

# Trip data
trip_record_bytes = 5_000       # route, timestamps, payment, ratings, etc.
daily_trip_bytes = uber_rides_per_day * trip_record_bytes
yearly_trip_bytes = daily_trip_bytes * 365

retention_years = 5

print(f"  Location updates:")
print(f"    Per update:            {location_update_bytes} bytes")
print(f"    Per day:               {fmt_bytes(daily_location_bytes)}")
print(f"    Per year:              {fmt_bytes(yearly_location_bytes)}")
print(f"    5-year total:          {fmt_bytes(yearly_location_bytes * retention_years)}")
print()
print(f"  Trip records:")
print(f"    Per trip:              {fmt_bytes(trip_record_bytes)}")
print(f"    Per day:               {fmt_bytes(daily_trip_bytes)}")
print(f"    Per year:              {fmt_bytes(yearly_trip_bytes)}")
print(f"    5-year total:          {fmt_bytes(yearly_trip_bytes * retention_years)}")
print()

# ── Step 5: Bandwidth ──
print("Step 5: Bandwidth")
print_divider()

# Location updates (writes)
location_bw_mbps = (location_qps_peak * location_update_bytes * 8) / 1e6

# Rider app: map tiles, ETA updates, driver position
rider_active_during_ride = uber_rides_per_day / (24 * 3600)  # concurrent riders
rider_app_kbps = 10  # ~10 Kbps for map updates during ride
rider_bw_mbps = (rider_active_during_ride * rider_app_kbps * 1000) / 1e6

print(f"  Location update bandwidth: {location_bw_mbps:,.1f} Mbps")
print(f"  Rider app bandwidth:       {rider_bw_mbps:,.1f} Mbps")
print(f"  Total peak bandwidth:      {location_bw_mbps + rider_bw_mbps:,.1f} Mbps")
print()

# ── Summary ──
print("📋 Uber Summary")
print("=" * 60)
print(f"  Location QPS (peak):    {location_qps_peak:>10,.0f}  ← Write-heavy!")
print(f"  Ride request QPS:       {ride_request_peak:>10,.0f}")
print(f"  Total API QPS (peak):   {total_api_qps:>10,.0f}")
print(f"  Location storage/year:  {fmt_bytes(yearly_location_bytes):>10}")
print(f"  Trip storage/year:      {fmt_bytes(yearly_trip_bytes):>10}")
print(f"  Peak bandwidth:         {location_bw_mbps + rider_bw_mbps:>8.0f} Mbps")
print()
print("  Key insight: Uber's challenge is WRITE THROUGHPUT (location updates)")
print("  and REAL-TIME processing (matching must happen in < 5 seconds).")
print("  Location data is written to a time-series DB, not regular Postgres.")

---

## 🎯 Estimation Tips for Interviews

### Do's ✅
1. **State your assumptions** — "I'll assume 200M DAU, which is typical for a large social platform"
2. **Round aggressively** — Use 100K instead of 86,400 for seconds/day. Close enough!
3. **Show the multiplication** — "200M users × 50 reads = 10B reads/day ÷ 100K seconds = 100K QPS"
4. **Sanity check** — "100K QPS would need ~10 Redis instances. That seems reasonable."
5. **Know when to stop** — Don't over-estimate. Get the big numbers, then move on to design.

### Don'ts ❌
1. **Don't memorize exact numbers** — Understand the order of magnitude
2. **Don't use outdated numbers** — Modern servers have 512 GB RAM, not 16 GB
3. **Don't over-shard** — A single Postgres handles 10K QPS and 10+ TB
4. **Don't skip the math** — Even if you know the answer, show the process
5. **Don't be precise** — "About 100K QPS" is better than "exactly 115,740.74 QPS"

### The 80/20 Rule
These three numbers get you through 80% of estimations:
```
1. Seconds in a day ≈ 100K
2. 1 million requests/day ≈ 12 QPS
3. A single server handles 10K-100K QPS (depending on operation)
```

In [ ]:
# Final comparison: all three systems side by side

print("📊 Comparison: Twitter vs YouTube vs Uber")
print("=" * 70)
print()
print(f"{'Metric':<25} {'Twitter':>15} {'YouTube':>15} {'Uber':>15}")
print("-" * 70)
print(f"  {'DAU':<23} {fmt(dau):>15} {fmt(yt_dau):>15} {fmt(uber_riders_mau):>15}")
print(f"  {'Read QPS (peak)':<23} {read_qps_peak:>13,.0f} {view_qps_peak:>13,.0f} {'N/A':>15}")
print(f"  {'Write QPS (peak)':<23} {write_qps_peak:>13,.0f} {upload_qps_avg:>13,.0f} {location_qps_peak:>13,.0f}")
print(f"  {'Storage/year':<23} {fmt_bytes(yearly_storage):>15} {fmt_bytes(yearly_upload_storage):>15} {fmt_bytes(yearly_location_bytes):>15}")
print(f"  {'Main challenge':<23} {'Fan-out':>15} {'Bandwidth':>15} {'Real-time':>15}")
print()
print("💡 Every system has a different bottleneck:")
print("   Twitter → Fan-out (delivering tweets to millions of followers)")
print("   YouTube → Bandwidth (streaming video to millions of viewers)")
print("   Uber    → Real-time writes (millions of location updates/sec)")
print()
print("   Knowing the numbers helps you identify the bottleneck BEFORE you design!")

---

## 🎯 Bonus: The Cache Hit Ratio Trick

The single most impactful number in capacity planning isn't QPS or storage — it's the
**cache hit ratio**. A 95% hit ratio means your database only sees 5% of the read traffic.
Get this wrong and you'll over-provision (or melt) your database.

Let's see how dramatic the difference is for our Twitter estimate.


In [ ]:
# Bonus: cache hit ratio drastically changes DB load
import math

# From our Twitter estimate
twitter_peak_read_qps = read_qps_peak  # tens of thousands of QPS
postgres_capacity = 10_000             # one Postgres handles ~10K reads/sec

print("How cache hit ratio changes Postgres fleet size")
print("=" * 60)
print(f"  Peak read QPS (timeline reads): {twitter_peak_read_qps:>10,.0f}")
print(f"  Per-Postgres read capacity:     {postgres_capacity:>10,}")
print()
print(f"  {'Cache hit %':<14} {'DB QPS':>12} {'Postgres replicas':>20}")
print("  " + "-" * 50)

for hit_ratio in [0.0, 0.50, 0.80, 0.90, 0.95, 0.99]:
    db_qps = twitter_peak_read_qps * (1 - hit_ratio)
    replicas = math.ceil(db_qps / postgres_capacity) if db_qps > 0 else 0
    print(f"  {hit_ratio:>12.0%}   {db_qps:>10,.0f}   {replicas:>15}")

print()
print("Lesson: going from 0% -> 95% cache hit ratio cuts DB infra by ~20x.")
print("This is why every serious read-heavy system has a cache layer.")
print()
print("Where the hit ratio comes from:")
print("  - Hot data is reused (Zipf distribution: top 20% of items = 80% of reads)")
print("  - Long TTLs on rarely-changing data")
print("  - Pre-warming caches for predictable hot keys (e.g. trending tweets)")


## 📚 Summary

### The Estimation Framework (memorize this!)

```
1. CLARIFY    → What are we estimating?
2. USERS      → MAU → DAU → actions per day
3. QPS        → daily_actions / 86,400 × 3 (peak)
4. STORAGE    → items/day × bytes/item × 365 × years
5. BANDWIDTH  → QPS × response_size
```

### Key Takeaways

1. **Every system has a different bottleneck** — Twitter = fan-out, YouTube = bandwidth, Uber = writes
2. **Show your math** — The process matters more than the exact answer
3. **Round aggressively** — 86,400 ≈ 100K, close enough for estimation
4. **Modern hardware is powerful** — Don't over-engineer based on 2015 numbers
5. **Sanity check your results** — If you need 1000 servers for a simple app, something is wrong

### Numbers to Take Into Your Interview

| What | Number |
|------|--------|
| Seconds/day | ~100K |
| 1M req/day | ~12 QPS |
| PostgreSQL QPS | ~10K (reads) |
| Redis QPS | ~100K |
| Server RAM | 256-512 GB |
| Server SSD | up to 60 TB |
| Network | 25 Gbps |

**You now have the tools to estimate capacity for ANY system. Go practice!** 🚀